# Prediction

Use existing databases to make predictions and perform interpolation.

In [1]:
import pprint

import cutcutcodec

from mendevi.models.nn import DNN
from mendevi.models.gpr import GPR
from mendevi.models.lr import LR

In [2]:
dataset = "svtav1_vs_rav1e_vs_aom.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Biaised linear regression

Predicts the output as a linear combination of the inputs, plus a constant.
It is a very fast model $O(n)$.

In [3]:
# model training
lr_model = LR(
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=["encoder"],  # optional, this feature provides one model per encoder
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster encoder=libaom-av1...
█ █ this cluster contains 2428 observations
█ █ 127.66 ms
█ Fit on cluster encoder=librav1e...
█ █ this cluster contains 2473 observations
█ █ 111.94 ms
█ Fit on cluster encoder=libsvtav1...
█ █ this cluster contains 2472 observations
█ █ 138.29 ms
█ 8.58 s


In [4]:
# validation from dataset
lr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")
# print(lr_model.accuracy)  # for each cluster name, associate for each output label, the predicted and validation data

Validate 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster encoder=libaom-av1...
█ █ this cluster contains 596 observations
█ █ 33.44 ms
█ Validate on cluster encoder=librav1e...
█ █ this cluster contains 615 observations
█ █ 51.39 ms
█ Validate on cluster encoder=libsvtav1...
█ █ this cluster contains 613 observations
█ █ 42.31 ms
█ 4.46 s
MAPE of log_rate: 3.54%
RMS  of log_rate: 0.2951
MAPE of psnr: 11.65%
RMS  of psnr: 6.709
MAPE of ssim: 4.17%
RMS  of ssim: 0.05084
MAPE of rate: 66.81%
RMS  of rate: 1.593e+07


In [5]:
# prediction from a video
pred = lr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Linear Regression for a predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 8.51 ms
█ 40.49 ms


{'log_rate': [6.372843265533447],
 'psnr': [34.864139556884766],
 'ssim': [0.9203274250030518]}


## Gaussian process regression

Very slow model $O(n^3)$. Efficient when data are very sparse.

In [6]:
# model training
gpr_model = GPR(  # no aggregation because it fits only on continuous parameters
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster effort=medium, encoder=libaom-av1...
█ █ this cluster contains 804 observations
█ █ 2 min 48 s
█ Fit on cluster effort=slow, encoder=libaom-av1...
█ █ this cluster contains 811 observations
█ █ 2 min 16 s
█ Fit on cluster effort=slow, encoder=librav1e...
█ █ this cluster contains 852 observations
/home/rrichard/.pyenv/versions/3.15.0rc2/envs/3.15/lib/python3.15/site-packages/sklearn/gaussian_process/_gpr.py:667: ConvergenceWarning: lbfgs failed to converge after 14 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
█ █ 3 min 29 s
█ Fit on cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 815 observations
█ █ 3 min 23 s
█ Fit on cluster effort=medium, encoder=librav1e...
█ █ this cluster contains 816 observations
█ █ 

In [7]:
# validation from dataset
gpr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster effort=fast, encoder=libaom-av1...
█ █ this cluster contains 201 observations
█ █ 28.87 ms
█ Validate on cluster effort=fast, encoder=librav1e...
█ █ this cluster contains 181 observations
█ █ 27.42 ms
█ Validate on cluster effort=slow, encoder=librav1e...
█ █ this cluster contains 208 observations
█ █ 28.74 ms
█ Validate on cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 196 observations
█ █ 27.92 ms
█ Validate on cluster effort=medium, encoder=libaom-av1...
█ █ this cluster contains 202 observations
█ █ 29.63 ms
█ Validate on cluster effort=medium, encoder=librav1e...
█ █ this cluster contains 226 observations
█ █ 29.09 ms
█ Validate on cluster effort=fast, encoder=libsvtav1...
█ █ this cluster contains 212 observations
█ █ 45.97 ms
█ Validate on cluster effort=slow, encoder=libsvtav1...
█ █ this cluster contains 205 observations
█ █ 39.32 ms


In [8]:
# prediction from a video
pred = gpr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Gaussian Process Regression predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 10.89 ms
█ 30.20 ms


{'log_rate': [6.327296733856201],
 'psnr': [40.00261306762695],
 'ssim': [0.9444530606269836]}


## Deep neuronal network

Medium speed model, $O(n)$.

In [9]:
# model training
dnn_model = DNN(  # see parameters niter, learning_rate and total
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=[],  # optional, this feature force a single common model
    output_labels=["log_rate", "psnr", "ssim"],
    niter=15000,  # number of training iterations, default 10000
    learning_rate=2e-5,  # adam gradient decay learning rate, default 1e-4
    total=25_000,  # number of model parameters, default based on number of observations
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 7373 observations
█ █ 18 min 43 s                                                                                                      
█ 18 min 50 s


In [10]:
# validation from dataset
dnn_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 1824 observations
█ █ 109.73 ms
█ 4.43 s
MAPE of log_rate: 2.78%
RMS  of log_rate: 0.226
MAPE of psnr: 6.80%
RMS  of psnr: 3.886
MAPE of ssim: 2.18%
RMS  of ssim: 0.0305
MAPE of rate: 44.42%
RMS  of rate: 1.342e+07


In [11]:
# prediction from a video
pred = dnn_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster ...
█ █ this cluster contains 1 observations
█ █ 13.12 ms
█ 40.61 ms


{'log_rate': [5.616203308105469],
 'psnr': [44.55204772949219],
 'ssim': [0.9506528377532959]}
